# Comprender los datos

In [1]:
# Importar paqueterías
import duckdb

# Conexión con el archivo de la base
con = duckdb.connect("../data/db/delitos_LA.duckdb", read_only=True)

## 1. Contexto y estructura

1. ¿Qué fenomeno registra la base?\
La base registra los delitos denunciados o conocidos por el departamento de policía de Los Angeles (LAPD), California durante el periodo 2020 a 2024.


2. ¿Quién la produjo y para qué?\
Si bien el registro pertenece al departamento de la policía de Los Angeles, el archivo se encuentra publicado por el gobierno de la ciudad mediante su portal de datos abiertos. Dentro de la página no hay una justificación de por qué se encuentra pública; sin embargo, dentro del ecosistema de datos abiertos, conocer la ocurrencia y ubicación de los delitos constituye un ejercicio de transparencia valioso para la ciudadanía, la academia y el sector empresarial.


3. ¿Cuántas filas y columnas contiene?\
El archivo contiene: 
- **filas:** 1,004,895
- **columnas:** 28


4. ¿Cuál es su cobertura temoral/geográfica?\
El archivo contiene delitos reportados entre 2020 y 2024, aunque para el análisis se utilizan únicamente los registros de 2020 a 2023 por la siguiente razón:

"On March 7th, 2024, the Los Angeles Police Department (LAPD) adopted a new Records Management System for reporting crimes and arrests. This new system is implemented to comply with the FBI's mandate to collect NIBRS-only data (NIBRS — FBI - https://www.fbi.gov/how-we-can-help-you/more-fbi-services-and-information/ucr/nibrs)."

Su cobertura geográfica abarca 21 áreas geográficas de la ciudad de Los Ángeles, California.


5. ¿Qué representa exactamente una fila (unidad de observación)?\
Cada fila representa un incidente delictivo (crime incident).


6. ¿Qué entidades, eventos e identificadores existen?\
La entidad corresponde al incidente delictivo reportado o conocido por LAPD. Cada incidente puede contener 1 o más delitos, registrados según su gravedad. El archivo cuenta con la columna DR_NO la cual representa el número de registro de la división. Este número contiene los dos dígitos del año, el identificador del área donde ocurrió el delito, y otros cinco dígitos. Asimismo, el archivo contiene un identificador para el área donde ocurrió el delito, representado por la columna 'AREA'. Por último, el archivo cuenta con la variable 'Crm Cd', la cual indica de manera numérica el tipo de delito reportado.


7. ¿Un identificador repetido representa necesariamente un error?\
Al explorar la base, no se encontró que para la variable 'DR_NO' existan registros repetidos. De haberlo, sí sería un error porque, conforme a la documentación de la base, cada fila representa a un único incidente. Para las variables 'AREA' y 'Crm Cd' sí puede repetirse, ya que cada delito pudo haberse cometido en una misma área de la ciudad y también un mismo delito. 


## 2. Calidad y decisiones de limpieza

8. Tipos de datos, faltantes y posibles valores inválidos.\
Para el esquema clean, decidimos trabajar con las siguientes variables:
- **fecha_occ:** TIMESTAMP
- **hora_occ:** INTEGER
- **diasem:** INTEGER
- **anio:** INTEGER
- **num_area:** INTEGER
- **nom_area:** VARCHAR
- **delito:** VARCHAR
- **sexo_vic:** VARCHAR
- **edad_vic:** INTEGER

Al explorar la base, encontramos algunos datos faltantes y valores inválidos, específicamente para las variables sexo de la víctima (sexo_vic) y edad de la víctima (edad_vic). 
Para sexo_vic, la documentación especifica que los valores de este campo son: F-female, M-male, y X-unknown. Sin embargo, aparecen otros valores como: '-' (1), 'H' (144) y 'NULL' (144,631).

Para la variable edad_vic, hay valores negativos que van del -4 al -1 (137), 120 (1). Igualmente, llama la atención la alta frecuencia del valor cero (269,178). La documentación no refiere sobre este campo en particular, por lo que dichos valores se tomarán como inválidos, ya que no corresponden con el tipo de dato o bien su frecuencia es atípica con respecto al resto de las edades registradas en el archivo. 

Finalmente, al revisar el número de registros por año, se evidencia una caída significativa en el número de registros para el año 2024, especialmente a partir de marzo. Al revisar la documentación del archivo, detectamos que el LAPD transitó a un diferente esquema de registro de incidentes delictivos (UCR a NIBRS). No obstante, no hay aclaraciones sobre por qué unos incidentes sí fueron registrados y otros no. Por esta razón, decidimos acotar el análisis para los registros que tuvieran como año de ocurrencia entre 2020 y 2023. 


9. Conversión de fechas y números; códigos y categorías inconsistentes.\ 
En el esquema clean construimos un solo campo de fecha de ocurrencia ('fecha_occ') que junta la fecha como la hora de ocurrencia del incidente delictivo. Esto se hizo mediante la siguiente sintaxis: 

```sql
(strptime("DATE OCC", '%m/%d/%Y %I:%M:%S %p')::DATE
    + strptime(lpad("TIME OCC", 4, '0'), '%H%M')::TIME) AS fecha_occ,
```

Asimismo, construimos dos campos derivados a partir de fecha_occ: día de la semana (diasem) y año (anio). 
La sintaxis para construir dichos campos fue la siguiente:

```sql
    EXTRACT(ISODOW FROM strptime("DATE OCC", '%m/%d/%Y %I:%M:%S %p'))::INTEGER AS diasem,
    EXTRACT(YEAR FROM strptime("DATE OCC", '%m/%d/%Y %I:%M:%S %p'))::INTEGER AS anio,
```

Para la variable de sexo_vict, se empleó la siguiente sintaxis a fin de visibilizar aquellos registros con los valores documentados por la base (F,M,X):

```sql
CASE WHEN "Vict Sex" IN ('F', 'M', 'X') THEN "Vict Sex" END AS sexo_vic,
```

Para la variable de edad_vic, se empleó la siguiente sintaxis:
```sql
CASE WHEN "Vict Age"::INTEGER BETWEEN 1 AND 99 THEN "Vict Age"::INTEGER END AS edad_vic,
```

En los registros que tengan valores diferentes a los especificados en la documentación para las variables sex_vic o edad_vic, aparecerá el valor 'NULL'. 
Así se mantienen todos los registros, pero a la vez se depura de mejor manera cuáles sí deben aparecer en las visualizaciones y cuáles no.


10. Posibles duplicados y reglas para tratarlos, cuando corresponda. 
En este caso no encontramos registros duplicados, por lo que no aplica un tratamiento distinto. 


11. La evidencia de las decisiones aplicadas en sql/02_clean.sql se muestra en los códigos a continuación

In [2]:
#sexo_vic

con.execute("""
    SELECT "Vict Sex", COUNT(*) AS n
    FROM raw.delitos_LA
    GROUP BY "Vict Sex"
    ORDER BY "Vict Sex"
""").df()

,Vict Sex,n
0,-,1
1,F,358553
2,H,114
3,M,403842
4,X,97753
5,NaN,144631


In [3]:
#edad_vic

con.execute("""
    SELECT "Vict Age", COUNT(*) AS n
    FROM raw.delitos_LA
    GROUP BY "Vict Age"
    ORDER BY "Vict Age"
""").df()

,Vict Age,n
0,-1,100
1,-2,28
2,-3,6
3,-4,3
4,0,269178
...,...,...
99,95,100
100,96,94
101,97,72
102,98,71


In [4]:
#anio

con.execute("""
    SELECT EXTRACT(YEAR FROM strptime("DATE OCC", '%m/%d/%Y %I:%M:%S %p')) AS anio,
           COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM raw.delitos_LA), 2) AS pct
    FROM raw.delitos_LA
    GROUP BY anio
    ORDER BY anio
""").df()

,anio,n,pct
0,2020,199847,19.89
1,2021,209876,20.89
2,2022,235259,23.41
3,2023,232345,23.12
4,2024,127567,12.69


## 3. Perfilado según el tipo de variable 

**El perfilado se realiza sobre clean.delitos_LA, que contiene los registros de 2020 a 2023.**

In [14]:
# Conteo de filas por capa y del universo de análisis (víctima mujer u hombre)

con.execute("""
    SELECT
        (SELECT COUNT(*) FROM raw.delitos_LA) AS filas_raw,
        (SELECT COUNT(*) FROM clean.delitos_LA) AS filas_clean,
        (SELECT COUNT(*) FROM clean.delitos_LA
         WHERE sexo_vic IN ('F', 'M')) AS filas_mujer_hombre,
        ROUND(100.0 * (SELECT COUNT(*) FROM clean.delitos_LA WHERE sexo_vic IN ('F', 'M'))
                    / (SELECT COUNT(*) FROM clean.delitos_LA), 1) AS pct_mujer_hombre
""").df()

,filas_raw,filas_clean,filas_mujer_hombre,pct_mujer_hombre
0,1004894,877327,683121,77.9


### 3.1 Categóricas

In [5]:
#sexo_vic

con.execute("""
    SELECT COALESCE(sexo_vic, 'Faltante') AS categoria,
           COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM clean.delitos_LA), 2) AS pct
    FROM clean.delitos_LA
    GROUP BY categoria
    ORDER BY n DESC
""").df()

,categoria,n,pct
0,M,360674,41.11
1,F,322447,36.75
2,Faltante,115529,13.17
3,X,78677,8.97


In [6]:
#nom_area

con.execute("""
    SELECT nom_area AS categoria,
           COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM clean.delitos_LA), 2) AS pct
    FROM clean.delitos_LA
    GROUP BY categoria
    ORDER BY n DESC
""").df()

,categoria,n,pct
0,Central,59456,6.78
1,77th Street,54981,6.27
2,Pacific,51351,5.85
3,Southwest,49127,5.60
4,Hollywood,46277,5.27
5,Southeast,44325,5.05
6,Olympic,44300,5.05
7,N Hollywood,43858,5.00
8,Newton,43787,4.99
9,Wilshire,41933,4.78


In [7]:
#delito

con.execute("""
    SELECT delito AS categoria,
           COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM clean.delitos_LA), 2) AS pct
    FROM clean.delitos_LA
    GROUP BY categoria
    ORDER BY n DESC
    LIMIT 21
""").df()

,categoria,n,pct
0,VEHICLE - STOLEN,93674,10.68
1,BATTERY - SIMPLE ASSAULT,69421,7.91
2,THEFT OF IDENTITY,56216,6.41
3,BURGLARY FROM VEHICLE,53990,6.15
4,BURGLARY,53411,6.09
5,"VANDALISM - FELONY ($400 & OVER, ALL CHURCH VA...",53141,6.06
6,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",49828,5.68
7,THEFT PLAIN - PETTY ($950 & UNDER),44898,5.12
8,INTIMATE PARTNER - SIMPLE ASSAULT,43717,4.98
9,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),33702,3.84


In [8]:
#num_area

con.execute("""
    SELECT num_area AS categoria,
           COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM clean.delitos_LA), 2) AS pct
    FROM clean.delitos_LA
    GROUP BY categoria
    ORDER BY categoria
""").df()

,categoria,n,pct
0,1,59456,6.78
1,2,41223,4.70
2,3,49127,5.60
3,4,32768,3.73
4,5,36085,4.11
5,6,46277,5.27
6,7,41933,4.78
7,8,40313,4.59
8,9,37122,4.23
9,10,36864,4.20


In [9]:
#diasem

con.execute("""
    SELECT diasem AS categoria,
           COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM clean.delitos_LA), 2) AS pct
    FROM clean.delitos_LA
    GROUP BY categoria
    ORDER BY categoria
""").df()

,categoria,n,pct
0,1,123716,14.10
1,2,120429,13.73
2,3,124702,14.21
3,4,123548,14.08
4,5,134094,15.28
5,6,128474,14.64
6,7,122364,13.95


In [10]:
#hora_occ

con.execute("""
    SELECT hora_occ AS categoria,
           COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM clean.delitos_LA), 2) AS pct
    FROM clean.delitos_LA
    GROUP BY categoria
    ORDER BY categoria
""").df()

,categoria,n,pct
0,0,36280,4.14
1,1,26390,3.01
2,2,22350,2.55
3,3,19531,2.23
4,4,16501,1.88
5,5,15257,1.74
6,6,19997,2.28
7,7,22618,2.58
8,8,32483,3.70
9,9,31848,3.63


In [11]:
#anio

con.execute("""
    SELECT anio AS categoria,
           COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM clean.delitos_LA), 2) AS pct
    FROM clean.delitos_LA
    GROUP BY categoria
    ORDER BY categoria
""").df()

,categoria,n,pct
0,2020,199847,22.78
1,2021,209876,23.92
2,2022,235259,26.82
3,2023,232345,26.48


### 3.2 Numéricas

In [12]:
#edad_vic

con.execute("""
    SELECT COUNT(*) AS total,
           COUNT(edad_vic) AS validos,
           COUNT(*) - COUNT(edad_vic) AS faltantes,
           ROUND(AVG(edad_vic), 2) AS media,
           MEDIAN(edad_vic) AS mediana,
           MODE(edad_vic) AS moda,
           MIN(edad_vic) AS minimo,
           MAX(edad_vic) AS maximo,
           MAX(edad_vic) - MIN(edad_vic) AS rango,
           ROUND(STDDEV_SAMP(edad_vic), 2) AS desv_est,
           ROUND(VAR_SAMP(edad_vic), 2) AS varianza,
           QUANTILE_CONT(edad_vic, 0.25) AS q1,
           QUANTILE_CONT(edad_vic, 0.75) AS q3
    FROM clean.delitos_LA
""").df()

,total,validos,faltantes,media,mediana,moda,minimo,maximo,rango,desv_est,varianza,q1,q3
0,877327,658739,218588,39.57,37.0,30,2,99,97,15.61,243.6,28.0,50.0


### 3.3 Temporales

In [13]:
#fecha_occ

con.execute("""
    SELECT COUNT(*) AS total,
           COUNT(fecha_occ) AS validos,
           COUNT(*) - COUNT(fecha_occ) AS faltantes,
           MIN(fecha_occ) AS fecha_min,
           MAX(fecha_occ) AS fecha_max
    FROM clean.delitos_LA
""").df()

,total,validos,faltantes,fecha_min,fecha_max
0,877327,877327,0,2020-01-01 00:01:00,2023-12-31 23:59:00


- La granularidad de este campo permite segmentar el análisis por día, mes, año, hora y minuto de ocurrencia.
- Su cobertura va del 1 de enero de 2020 hasta el 31 de diciembre de 2023.

### 3.4 Geográficas

La única variable geográfica para este análisis será nom_area que representa el nombre del área donde ocurrió el incidente delictivo. 
Si bien el archivo contiene coordenadas (lat,long), éstas no serán utilizadas porque no se hará un análisis geoespacial de los delitos. 
Así, dicha variable cubre las 21 zonas geográficas que LAPD divide a la ciudad de Los Angeles. 

## 4. Implicaciones para el análisis

##### Preguntas viables
Conforme al esquema que definimos para este ejercicio, las preguntas son de tipo descriptivas. Nos interesa particularmente conocer qué delitos afectan más a mujeres y a hombres. Por la cobertura temporal del archivo, será interesante tomar en cuenta si hubo diferencias entre años, ya que el periodo analizado coincide con la duración de la pandemia por COVID-19. Asimismo, nos interesa identificar si todas las áreas de la ciudad presentan una distribución similar de delitos o si bien hay áreas que concentran más delitos que otras. Lo anterior puede servir como dato orientador para priorizar los recursos de LAPD. 
Otras preguntas que son viables para el estudio son: ¿los delitos más frecuentes para mujeres y hombres se mantuvieron en el mismo lugar entre años? ¿qué días de la semana y horas son los de mayor riesgo para los delitos de mayor frecuencia? 


##### Sesgos de cobertura

Considerando que el análisis se centrará únicamente en aquellos incidentes que tuvieron un correcto registro, es posible que se estén omitiendo para la variable sexo_vic a ciertas víctimas para las cuales no fue posible identificar su sexo. 
Asimismo, la alta frecuencia del valor cero en la variable edad_vic puede potencialmente sesgar el análisis. No hay certeza que efectivamente se trata de delitos para personas menores de un año o si más bien se trata de registros donde no fue posible identificar la edad de la víctima o bien donde no aplicara. 

##### Precauciones al interpretar los resultados

Es importante considerar que el análisis utilizará incidentes delictivos reportados a LAPD. Esto significa que no se trata de la totalidad de delitos ocurridos en Los Angeles. Si se quisiera entender la magnitud real del problema delictivo, entonces se tendría que realizar un análisis a partir de encuestas de victimización con desagregación local. Este análisis únicamente visualiza los patrones de aquellos eventos que fueron del conocimiento de la autoridad, por lo que se debe tomar esta cautela y los hallazgos contrastarlos con los datos de las encuestas de victimización. 